# CHARIS-DPP in Python: tutorial

A complete reduction on a synthetic dataset (runs without real data or network). Each function below is also a `charis <step>` command.

In [ ]:
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
# plots are saved to files by the pipeline; this shows them inline too
%matplotlib inline

## 1. Make a dataset

With real data, run `charis newobs DATE OBJECT BAND` in the directory with your `CRSA*_cube.fits` instead.

In [ ]:
from charis_dpp.synthetic import SyntheticConfig, make_dataset

work = Path("tutorial_synth").resolve()
cfg_path = make_dataset(
    work,
    SyntheticConfig(nframes=8, shape=(101, 101), n_speckles=12, nsky=2, seed=3, astrogrid_lambda_d=11.2),
)
print(cfg_path)
print(sorted(p.name for p in (work / "data" / "raw").iterdir())[:4], "...")

## 2. Configuration

Constants come from the package's dated instrument table; the dataset `.toml` holds what is specific to this observation; keyword arguments override for one call.

In [ ]:
from charis_dpp.config.dataset import DatasetConfig

cfg = DatasetConfig.load(cfg_path)
res = cfg.resolve()
print("band", res.band, "| channels", res.nlam, "| reference channel", res.reference_channel)
print("instrument epoch:", res.instrument.epoch_label, "| pixscale", res.instrument.pixscale_arcsec,
      "| north offset", res.instrument.north_offset_deg, "deg")
print("telluric channels:", res.telluric_channels.tolist())
print("frames:", cfg.frames("science"), "| sky:", cfg.frames("sky"))
print(cfg_path.read_text()[:600])

An `[instrument]` table in the `.toml` overrides a constant for this dataset only.

## 3. Prep, sky subtraction, registration

In [ ]:
from charis_dpp.prelims.imprep import imprep
from charis_dpp.skysub import subtract_sky
from charis_dpp.reg import register_cube

obslog = imprep(cfg_path)            # reduc/prep/n####e.fits, obslog.txt, frame lists in the .toml
subtract_sky(cfg_path)               # reduc/prep/n####e_skysub.fits
reduclog = register_cube(cfg_path)   # reduc/reg/n####reg.fits + reduc.log
print(open(work / "reduc.log").read()[:400])

Registration uses the satellite spots by default. Without spots, use `method="xcorr"`.

In [ ]:
import json
truth = json.loads((work / "truth.json").read_text())
true_centres = np.array([f["center"] for f in truth["frames"]])
meas = reduclog.centers(cfg.frames("science"))
print("registration error (px), max:", np.abs(meas - true_centres).max().round(3))

# the spot-free alternative (writes the same products; here just to show the call)
# register_cube(cfg_path, method="xcorr", xcorr_center="centroid")

## 4. Flux calibration and PSF model

In [ ]:
from charis_dpp.specphotcal import specphot_cal, make_empirical_psf

specphot_cal(cfg_path, calmethod=1, starlib="pickles")     # reduc/reg/n####reg_cal.fits (mJy)
make_empirical_psf(cfg_path)                               # psfmodel/psfcube_med.fits
h = fits.getheader(work / "reduc" / "reg" / "n0001reg_cal.fits", 1)
print("FLUXUNIT", h["FLUXUNIT"], "| FSTAR_10 =", h["FSTAR_10"], "| R_AP10 =", h["R_AP10"])

## 5. Filtering and A-LOCI

In [ ]:
from charis_dpp.filt import imrsub
from charis_dpp.psfsub import adialoci

imrsub(cfg_path, prad=True)                                 # radial-profile subtraction -> reduc/rsub
final, collapsed = adialoci(
    cfg_path, rsubval=1, nfwhm=0.75, drsub=5, na=100, svd=1e-6, nref=0, rmin=6, rmax=40, outfile="aloci.fits"
)
img = fits.getdata(work / "reduc" / "proc" / "aloci_collapsed.fits")
plt.figure(figsize=(5, 5))
plt.imshow(img, origin="lower", vmin=np.nanpercentile(img, 1), vmax=np.nanpercentile(img, 99.7), cmap="magma")
p = truth["planet"]
plt.plot(p["x_northup"], p["y_northup"], "o", mfc="none", mec="cyan", ms=14)
plt.title("A-LOCI, collapsed (circle: injected planet)")
plt.show()

## 6. Detection, forward modelling, spectrum

In [ ]:
from charis_dpp.tools.detection import find_detections
from charis_dpp.partialsub.fwdmod_planet import fwdmod_planet
from charis_dpp.partialsub.inject import Planet
from charis_dpp.specphotcal.extract_spectrum import extract_spectrum

dets = find_detections(cfg_path, "aloci.fits", outfile=work / "reduc" / "proc" / "aloci_det.targ")
best = dets[0]
print(f"detection at ({best.x:.1f}, {best.y:.1f}) SNR {best.snr:.1f}; truth ({p['x_northup']:.1f}, {p['y_northup']:.1f})")

thr = fwdmod_planet(cfg_path, reducname="aloci.fits", planets=[Planet(best.x, best.y, best.log_contrast)], psf="empirical")
spec = extract_spectrum(cfg_path, "aloci.fits", (best.x, best.y), throughput="synth_throughput0.txt")

fstar = np.array([h[f"FSTAR_{l}"] for l in range(res.nlam)])
plt.figure(figsize=(6, 4))
plt.errorbar(spec.wavelength_um, spec.flux / fstar, spec.err / fstar, fmt="o", label="recovered contrast")
plt.plot(spec.wavelength_um, p["contrast"], "-", label="injected")
plt.plot(spec.wavelength_um, thr[0] * np.array(p["contrast"]), ":", label="injected x throughput")
plt.xlabel("wavelength [um]"); plt.ylabel("planet/star"); plt.legend(); plt.show()

## 7. Attenuation map and contrast curve

In [ ]:
from charis_dpp.partialsub.attenmap import attenuation_map
from charis_dpp.tools.contrast import final_contrast

attenuation_map(cfg_path, reducname="aloci.fits", ntc=1, lrmin=8, lrmax=36, dlrstep=4)
c = final_contrast(cfg_path, "aloci.fits")
plt.figure(figsize=(6, 4))
plt.semilogy(c.sep_arcsec, c.broadband, label="broadband 5-sigma")
for name, curve in c.bands.items():
    plt.semilogy(c.sep_arcsec, curve, lw=0.8, label=name)
plt.xlabel("separation [arcsec]"); plt.ylabel("contrast"); plt.legend(); plt.show()

## 8. PDI in brief

Two-beam synthetic data with a polarised ring: the ring must show in $Q_\phi$, not $U_\phi$.

In [ ]:
from charis_dpp.pdi import hwp_match, pdi, pdi_register_cube, pdi_specphot_cal, pdi_subtract_sky

pwork = Path("tutorial_pdi").resolve()
pcfg = make_dataset(pwork, SyntheticConfig(nframes=8, shape=(161, 161), n_speckles=8, nsky=2, seed=3,
                                           astrogrid_lambda_d=11.2, pdi=True, planet=None, pa_step_deg=3.0))
imprep(pcfg, pdi=True)
pdi_subtract_sky(pcfg)
pdi_register_cube(pcfg)                        # left/right beams -> n####{left,right}reg.fits
pdi_specphot_cal(pcfg, calmethod=1, starlib="pickles")
table = hwp_match(pcfg)                        # from vampHWPLog_*.csv; source="header" uses RET-ANG1
print("complete HWP cycles:", table.ncycles)
r = pdi(pcfg, mm_ip_corr=False, mm_crosstalk_corr=False, adisync=False, bg_sub=True, plot=False)
qphi = fits.getdata(pwork / "reduc" / "proc" / "pdi_qphi_collapsed.fits")
uphi = fits.getdata(pwork / "reduc" / "proc" / "pdi_uphi_collapsed.fits")
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
v = np.nanpercentile(qphi, 99.5)
ax[0].imshow(qphi, origin="lower", vmin=-v, vmax=v, cmap="RdBu_r"); ax[0].set_title("Q_phi")
ax[1].imshow(uphi, origin="lower", vmin=-v, vmax=v, cmap="RdBu_r"); ax[1].set_title("U_phi")
plt.show()

Keep the Mueller-matrix corrections on with real data (the defaults).

Next: `docs/WORKFLOW.md`, `charis <step> --help`, `tests/e2e/` for more chains.